In [1]:
# 1. Dynamically download python storage dependencies right into the running container
!pip install --no-cache-dir boto3 pyarrow

In [2]:
import io
import boto3
import pandas as pd
from botocore.client import Config

# 2. Connect to the storage container
s3 = boto3.client(
    "s3",
    endpoint_url="http://storage:9000",
    aws_access_key_id="root",
    aws_secret_access_key="password",
    config=Config(signature_version="s3v4"),
    region_name="us-east-1",
)

# 3. Pull the specific file from your bronze data lake pool
obj = s3.get_object(Bucket="bronze", Key="101.parquet")
df = pd.read_parquet(io.BytesIO(obj["Body"].read()))

print("--- SCHEMA BLUEPRINT ---")
print(df.info())

print("\n--- DATA PREVIEW ---")
print(df.head(5))

--- SCHEMA BLUEPRINT ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1686407 entries, 0 to 1686406
Data columns (total 11 columns):
 #   Column               Non-Null Count    Dtype         
---  ------               --------------    -----         
 0   listened_at          1686407 non-null  datetime64[ns]
 1   created              1686407 non-null  datetime64[ns]
 2   user_id              1686407 non-null  int64         
 3   recording_msid       1686407 non-null  object        
 4   artist_name          1686407 non-null  object        
 5   artist_credit_id     1529895 non-null  float64       
 6   release_name         1671229 non-null  object        
 7   release_mbid         1529594 non-null  object        
 8   recording_name       1686407 non-null  object        
 9   recording_mbid       1538198 non-null  object        
 10  artist_credit_mbids  1529900 non-null  object        
dtypes: datetime64[ns](2), float64(1), int64(1), object(7)
memory usage: 141.5+ MB
None

--- DA

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql.functions import to_timestamp

# Kill old context so the JVM can reload fresh packages from spark-defaults.conf
try: spark.stop()
except: pass

spark = SparkSession.builder.getOrCreate()

# Load your file now!
feedback_schema = StructType([
    StructField("feedback_id", IntegerType(), True),
    StructField("user_id", IntegerType(), True),
    StructField("recording_msid", StringType(), True),
    StructField("recording_mbid", StringType(), True),
    StructField("score", IntegerType(), True),
    StructField("created_at", StringType(), True) 
])

feedback_df = (spark.read \
    .option("delimiter", "\t") \
    .option("nullValue", "\\N") \
    .schema(feedback_schema) \
    .csv("s3a://bronze/metadata-dump/listenbrainz-public-dump-20260415-000003/lbdump/recording_feedback")
)

feedback_df = feedback_df.withColumn("created_at", to_timestamp("created_at"))
feedback_df.show(5, truncate=False)
feedback_df.show(5, truncate=False)

In [12]:
import os

conf_path = "spark-defaults.conf"

print("--- DECODING YOUR TRUE SPARK CONFIGURATION ---")
if os.path.exists(conf_path):
    with open(conf_path, "r") as f:
        for line in f:
            # Print master, host, endpoint, or catalog configs
            if line.strip() and not line.startswith("#"):
                if any(k in line for k in ["master", "endpoint", "catalog", "warehouse"]):
                    print(line.strip())
else:
    print(f"Could not find {conf_path} in current directory.")

--- DECODING YOUR TRUE SPARK CONFIGURATION ---
spark.hadoop.fs.s3a.endpoint=http://storage:9000
spark.sql.catalog.nessie=org.apache.iceberg.spark.SparkCatalog
spark.sql.catalog.nessie.catalog-impl=org.projectnessie.iceberg.v2.NessieIcebergCatalog
spark.sql.catalog.nessie.uri=http://nessie:19120/api/v2
spark.sql.catalog.nessie.ref=main
spark.sql.catalog.nessie.authentication.type=NONE
spark.sql.catalog.nessie.warehouse=s3a://silver/
spark.sql.catalog.nessie.io-impl=org.apache.iceberg.aws.s3.S3FileIO
spark.sql.catalog.nessie.s3.access-key-id=root
spark.sql.catalog.nessie.s3.secret-access-key=password
spark.sql.catalog.nessie.s3.endpoint=http://storage:9000
spark.sql.catalog.nessie.s3.region=us-east-1
spark.sql.catalog.nessie.s3.path-style-access=true


In [ ]:
import os

conf_path = "spark-defaults.conf"

print("--- DECODING YOUR TRUE SPARK CONFIGURATION ---")
if os.path.exists(conf_path):
    with open(conf_path, "r") as f:
        for line in f:
            # Print master, host, endpoint, or catalog configs
            if line.strip() and not line.startswith("#"):
                if any(k in line for k in ["master", "endpoint", "catalog", "warehouse"]):
                    print(line.strip())
else:
    print(f"Could not find {conf_path} in current directory.")